# Смесевые модели и EM-алгоритм: поиск скрытых подпопуляций

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Смесевые модели и EM-алгоритм».

## Что делает метод

Смесевая модель отвечает на вопрос: «не является ли моя выборка на самом деле несколькими перекрывающимися группами?» Если гистограмма признака показывает два или более горба, за ней может стоять смесь подпопуляций: пациенты с разным течением болезни, образцы из разных литологических фаций, клетки разных типов по профилю экспрессии.

Смесь гауссовых распределений (GMM, Gaussian Mixture Model) — наиболее распространённая версия: каждая скрытая подпопуляция описывается своим нормальным распределением. В отличие от жёсткой кластеризации (K-средних), GMM даёт каждому наблюдению **вектор вероятностей принадлежности** ко всем компонентам — это называют мягкой кластеризацией. Параметры модели оцениваются EM-алгоритмом.

В этом блокноте мы подгоним GMM к данным, сравним её с K-средними, выберем число компонент по BIC и разберём, как читать результат. Расчётное время прохождения — около пяти минут. GPU не требуется.

## Интуиция за методом

Представьте гистограмму суточного потребления калорий в когорте, включающей людей с нормальным обменом и людей с метаболическим синдромом. Вы увидите два горба, а не один: две подпопуляции с разными средними и разбросами наложились друг на друга. Смесевая модель формализует эту идею: итоговое распределение — это взвешенная сумма нескольких компонент, каждая со своими параметрами.

EM-алгоритм находит параметры через два попеременных шага:

- **E-шаг (Expectation)**: при зафиксированных параметрах компонент вычислить вероятность того, что каждое наблюдение порождено каждой компонентой (апостериорные вероятности принадлежности — responsibility).
- **M-шаг (Maximization)**: при зафиксированных responsibility обновить средние, ковариации и веса компонент взвешенными оценками.

Итерации продолжаются до сходимости. Логарифмическое правдоподобие монотонно растёт на каждом шаге — сходимость гарантирована теорией, но лишь к **локальному** оптимуму, откуда важность нескольких рестартов.

**Ключевые понятия, которые встретятся в блокноте:**

- **Компонента** — одно из слагаемых смеси; в GMM это нормальное распределение со своим средним и ковариационной матрицей.
- **Весовая доля (mixing weight)** — доля наблюдений, порождённых данной компонентой; сумма весов по всем компонентам равна 1.
- **Responsibility (мягкая принадлежность)** — вероятность того, что данное наблюдение принадлежит данной компоненте; сумма по компонентам равна 1.
- **BIC (байесовский информационный критерий)** — штрафует правдоподобие за сложность модели (число компонент); выбирают модель с наименьшим BIC.
- **Ковариационная структура** — форма разрешённых эллипсоидов компонент: `full` (произвольные), `diag` (выровненные по осям), `tied` (общая матрица), `spherical` (изотропные).

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует нужные версии, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn==1.5.1 numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.2 scipy==1.13.1

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Для демонстрации используем синтетическую смесь трёх гауссовых компонент в двух измерениях. Такой выбор позволяет контролировать истинные параметры и наглядно оценить точность восстановления. Дополнительно покажем одномерную проекцию с видимыми горбами гистограммы — именно с такой картиной исследователь сталкивается в реальных данных, когда замечает неоднородность.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Воспроизводимость: все случайные состояния зафиксированы.
RNG = np.random.default_rng(42)

# Истинные параметры трёх компонент смеси.
TRUE_MEANS = np.array([
    [-4.0,  1.0],   # компонента 1: меньшие значения
    [ 0.5,  4.0],   # компонента 2: средние значения
    [ 4.0, -1.5],   # компонента 3: большие значения
])
TRUE_COVS = [
    np.array([[1.0, 0.6], [0.6, 0.8]]),   # вытянутый эллипс
    np.array([[0.7, 0.0], [0.0, 1.5]]),   # вертикальный эллипс
    np.array([[1.2, -0.5], [-0.5, 0.9]]), # наклонённый эллипс
]
TRUE_WEIGHTS = np.array([0.35, 0.40, 0.25])   # весовые доли, сумма = 1
N_SAMPLES = 600

# Генерируем смесь: для каждой точки выбираем компоненту по весам,
# затем сэмплируем из соответствующего нормального распределения.
component_ids = RNG.choice(3, size=N_SAMPLES, p=TRUE_WEIGHTS)
X_data = np.vstack([
    RNG.multivariate_normal(TRUE_MEANS[k], TRUE_COVS[k])
    for k in component_ids
])
true_labels = component_ids   # только для оценки — в реальной задаче их нет

df = pd.DataFrame(X_data, columns=["x1", "x2"])
print(f"Сгенерировано {N_SAMPLES} наблюдений, 2 признака.")
print(f"Истинные весовые доли: {TRUE_WEIGHTS}")
df.describe().round(3)

In [ ]:
# Наглядный «ага»-момент: данные без разметки и признак неоднородности.
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0))

# Левый: облако точек без цветовой разметки — именно так выглядят данные
# до применения метода.
axes[0].scatter(X_data[:, 0], X_data[:, 1],
                s=18, alpha=0.45, color=VIZ["edge"])
axes[0].set_title("Данные до анализа\n(разметки компонент нет)")
axes[0].set_xlabel("Признак 1")
axes[0].set_ylabel("Признак 2")

# Правый: гистограмма первого признака показывает несколько горбов —
# первый сигнал неоднородности выборки.
axes[1].hist(X_data[:, 0], bins=40, color=VIZ["series"][0],
             alpha=0.75, edgecolor="none")
axes[1].set_title("Гистограмма признака 1\n(видны горбы — возможна смесь)")
axes[1].set_xlabel("Признак 1")
axes[1].set_ylabel("Частота")

fig.tight_layout()
plt.show()
print("Гистограмма показывает три горба — вероятно, три подпопуляции.")

## 4. Применение метода

### Шаг 4.1. Подгонка GMM и просмотр оценённых параметров

Класс `GaussianMixture` из scikit-learn реализует EM-алгоритм. Ключевые параметры:

- `n_components` — число компонент (задаётся вручную; далее покажем, как подобрать его по BIC).
- `covariance_type` — тип ковариационной матрицы; `"full"` — наиболее гибкий вариант.
- `n_init` — число рестартов с разной инициализацией; выбирается решение с наибольшим правдоподобием.
- `random_state` — фиксация воспроизводимости.

In [ ]:
from sklearn.mixture import GaussianMixture

# Подгоняем GMM с тремя компонентами и 10 рестартами.
# n_init=10 снижает риск попасть в плохой локальный оптимум.
gmm = GaussianMixture(
    n_components=3,
    covariance_type="full",
    n_init=10,
    random_state=42,
)
gmm.fit(X_data)

print(f"Сошлось за {gmm.n_iter_} итераций EM.")
print(f"Логарифмическое правдоподобие: {gmm.lower_bound_:.2f}")
print()
print("Оценённые весовые доли компонент:")
for k, w in enumerate(gmm.weights_):
    print(f"  Компонента {k+1}: {w:.3f}  (истинная: {TRUE_WEIGHTS[k]:.3f})")
print()
print("Оценённые средние компонент:")
for k, mu in enumerate(gmm.means_):
    print(f"  Компонента {k+1}: [{mu[0]:+.3f}, {mu[1]:+.3f}]  "
          f"(истинное: [{TRUE_MEANS[k,0]:+.1f}, {TRUE_MEANS[k,1]:+.1f}])")

### Шаг 4.2. Мягкие вероятности принадлежности (responsibility)

Метод `predict_proba()` возвращает матрицу размером (n_samples × n_components): каждая строка — вектор вероятностей принадлежности одного наблюдения к каждой компоненте. Это принципиально отличает GMM от K-средних, которые присваивают жёсткую метку. Наблюдения в области перекрытия компонент получают неопределённые вероятности — что честно отражает реальную неоднозначность.

In [ ]:
# Получаем мягкие вероятности принадлежности.
responsibilities = gmm.predict_proba(X_data)   # форма: (600, 3)
hard_labels_gmm = gmm.predict(X_data)          # жёсткие метки по argmax

print("Первые 10 строк матрицы responsibility (GMM):")
print("  Компон.1   Компон.2   Компон.3")
for row in responsibilities[:10]:
    print("  " + "   ".join(f"{v:.4f}" for v in row))

# Сколько наблюдений имеют «неопределённую» принадлежность —
# вероятность лидирующей компоненты меньше 0.95?
uncertain = (responsibilities.max(axis=1) < 0.95).sum()
print(f"\nНаблюдений с неопределённой принадлежностью (p_max < 0.95): "
      f"{uncertain} из {N_SAMPLES} ({100*uncertain/N_SAMPLES:.1f}%)")

### Шаг 4.3. GMM против K-средних: сравнение мягкой и жёсткой кластеризации

Два графика рядом показывают принципиальную разницу. На левом — K-средних: каждая точка одного цвета, граница проходит там, где метод «разрезает» пространство. На правом — GMM: цвет точки отражает вероятность принадлежности к доминирующей компоненте (насыщенность соответствует уверенности), а точки в зонах перекрытия выглядят менее насыщенными.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# K-средних для сравнения.
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
hard_labels_km = kmeans.fit_predict(X_data)

palette = get_palette(3)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))

# --- Левый: K-средних (жёсткая кластеризация) ---
for c in range(3):
    mask = hard_labels_km == c
    axes[0].scatter(X_data[mask, 0], X_data[mask, 1],
                    s=20, alpha=0.65, color=palette[c],
                    label=f"Кластер {c+1}")
# Центроиды K-средних
axes[0].scatter(kmeans.cluster_centers_[:, 0],
                kmeans.cluster_centers_[:, 1],
                s=200, color="white", edgecolors=VIZ["node_text"],
                linewidths=2, zorder=5, marker="*")
axes[0].set_title("K-средних: жёсткая кластеризация")
axes[0].set_xlabel("Признак 1")
axes[0].set_ylabel("Признак 2")
axes[0].legend(loc="upper right", fontsize=10)

# --- Правый: GMM (мягкая кластеризация) ---
# Цвет — доминирующая компонента; alpha — уверенность модели (max probability).
dominant = responsibilities.argmax(axis=1)
confidence = responsibilities.max(axis=1)   # от 0 до 1

for c in range(3):
    mask = dominant == c
    axes[1].scatter(X_data[mask, 0], X_data[mask, 1],
                    s=20, alpha=confidence[mask] * 0.85 + 0.10,
                    color=palette[c], label=f"Компонента {c+1}")
# Оценённые средние GMM
axes[1].scatter(gmm.means_[:, 0], gmm.means_[:, 1],
                s=200, color="white", edgecolors=VIZ["node_text"],
                linewidths=2, zorder=5, marker="*")
axes[1].set_title("GMM: мягкая кластеризация\n(прозрачность = неопределённость)")
axes[1].set_xlabel("Признак 1")
axes[1].set_ylabel("Признак 2")
axes[1].legend(loc="upper right", fontsize=10)

fig.tight_layout()
plt.show()

**Как читать эти графики.**

- **K-средних (левый)**: жёсткое разбиение — каждая точка принадлежит ровно одному кластеру. Граница между кластерами проходит там, где метод «разрезает» пространство перпендикулярно линии, соединяющей центроиды. Точки на границе получают метку произвольно.
- **GMM (правый)**: мягкая кластеризация. Звёздочки — оценённые средние компонент. Точки в центрах компонент насыщены по цвету (уверенность близка к 1); точки в зонах перекрытия полупрозрачны — модель честно отражает неопределённость принадлежности. GMM также учитывает форму и ориентацию каждого облака (ковариационную матрицу), что позволяет корректно описывать вытянутые и наклонённые кластеры, с которыми K-средних справляется хуже.

### Шаг 4.4. Выбор числа компонент по BIC

EM-алгоритм не определяет число компонент K автоматически. Для его выбора обучают серию моделей при K = 1, 2, …, K_max и смотрят на **BIC (байесовский информационный критерий)**:

> BIC = −2 · ln(L) + p · ln(n)

где L — максимальное правдоподобие, p — число параметров модели, n — число наблюдений. Штраф за сложность растёт с числом компонент. Выбирают K с **наименьшим** BIC.

In [ ]:
K_candidates = range(1, 9)
bic_scores = []
aic_scores = []

for k in K_candidates:
    m = GaussianMixture(n_components=k, covariance_type="full",
                        n_init=5, random_state=42)
    m.fit(X_data)
    bic_scores.append(m.bic(X_data))
    aic_scores.append(m.aic(X_data))

best_k_bic = list(K_candidates)[int(np.argmin(bic_scores))]
best_k_aic = list(K_candidates)[int(np.argmin(aic_scores))]
print(f"Оптимальное K по BIC: {best_k_bic}  (истинное: 3)")
print(f"Оптимальное K по AIC: {best_k_aic}  (истинное: 3)")

# График BIC и AIC по числу компонент.
fig, ax = plt.subplots(figsize=(9.5, 5.0))

ax.plot(list(K_candidates), bic_scores, marker="o",
        color=VIZ["series"][0], label="BIC")
ax.plot(list(K_candidates), aic_scores, marker="s",
        color=VIZ["series"][1], linestyle="--", label="AIC")
ax.axvline(best_k_bic, color=VIZ["series"][2], linestyle=":",
           linewidth=1.8, label=f"Минимум BIC: K={best_k_bic}")

ax.set_title("Выбор числа компонент GMM по информационным критериям")
ax.set_xlabel("Число компонент K")
ax.set_ylabel("Значение критерия (меньше — лучше)")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

**Как читать этот график.**

Обе кривые убывают при увеличении K, пока рост правдоподобия компенсирует штраф за параметры. В точке минимума штраф начинает доминировать. BIC штрафует строже (пропорционально ln(n)), поэтому минимум BIC обычно при меньшем K, чем у AIC — BIC более консервативен. Практический совет: ищите минимум BIC; если кривая имеет широкое «плато», доверяйте скорее содержательной интерпретируемости, чем формальному минимуму.

### Шаг 4.5. Влияние числа рестартов на результат

EM сходится к локальному оптимуму. Ячейка ниже демонстрирует разброс итоговых правдоподобий при одном рестарте против десяти.

In [ ]:
N_REPEATS = 20  # число независимых прогонов для оценки разброса

ll_1init  = []  # логарифм правдоподобия при n_init=1
ll_10init = []  # логарифм правдоподобия при n_init=10

for seed in range(N_REPEATS):
    m1 = GaussianMixture(n_components=3, covariance_type="full",
                         n_init=1, random_state=seed)
    m1.fit(X_data)
    ll_1init.append(m1.lower_bound_)

    m10 = GaussianMixture(n_components=3, covariance_type="full",
                          n_init=10, random_state=seed)
    m10.fit(X_data)
    ll_10init.append(m10.lower_bound_)

ll_1init  = np.array(ll_1init)
ll_10init = np.array(ll_10init)

print("Разброс логарифмического правдоподобия по 20 прогонам:")
print(f"  n_init=1:   min={ll_1init.min():.3f}  max={ll_1init.max():.3f}  "
      f"std={ll_1init.std():.4f}")
print(f"  n_init=10:  min={ll_10init.min():.3f}  max={ll_10init.max():.3f}  "
      f"std={ll_10init.std():.4f}")

fig, ax = plt.subplots(figsize=(9.5, 4.5))
ax.plot(ll_1init, marker="o", color=VIZ["series"][2],
        alpha=0.8, label="n_init=1 (один рестарт)")
ax.plot(ll_10init, marker="s", color=VIZ["series"][0],
        alpha=0.8, label="n_init=10 (десять рестартов)")
ax.set_title("Влияние числа рестартов на итоговое правдоподобие")
ax.set_xlabel("Прогон (случайное начальное состояние)")
ax.set_ylabel("Логарифмическое правдоподобие (выше — лучше)")
ax.legend()
fig.tight_layout()
plt.show()

### Шаг 4.6. Сравнение типов ковариационной структуры

`GaussianMixture` поддерживает четыре варианта ковариации. Ячейка сравнивает их BIC и визуально показывает, как тип ковариации меняет форму подобранных компонент.

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
import numpy as np

COV_TYPES = ["full", "tied", "diag", "spherical"]
COV_LABELS = {
    "full":      "full — произвольный эллипс",
    "tied":      "tied — общая матрица для всех",
    "diag":      "diag — оси вдоль координат",
    "spherical": "spherical — окружность",
}

gmm_variants = {}
bic_variants = {}
for ct in COV_TYPES:
    m = GaussianMixture(n_components=3, covariance_type=ct,
                        n_init=10, random_state=42)
    m.fit(X_data)
    gmm_variants[ct] = m
    bic_variants[ct] = m.bic(X_data)

print("BIC по типу ковариации (K=3):")
for ct in COV_TYPES:
    print(f"  {ct:12s}: {bic_variants[ct]:.1f}"
          + ("  <- минимум" if bic_variants[ct] == min(bic_variants.values()) else ""))

# Вспомогательная функция: рисует эллипс компоненты по её ковариационной матрице.
def draw_ellipse(ax, mean, cov_matrix, color, n_std=2.0, alpha=0.18):
    """Рисует эллипс доверия уровня n_std стандартных отклонений."""
    vals, vecs = np.linalg.eigh(cov_matrix)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=mean, width=w, height=h, angle=theta,
                  facecolor=color, alpha=alpha, edgecolor=color,
                  linewidth=1.8, linestyle="-")
    ax.add_patch(ell)


# Графики по четырём типам ковариации.
palette = get_palette(3)
fig, axes = plt.subplots(2, 2, figsize=(13.5, 11.0))
axes_flat = axes.flatten()

for ax, ct in zip(axes_flat, COV_TYPES):
    m = gmm_variants[ct]
    labels_ct = m.predict(X_data)

    for c in range(3):
        mask = labels_ct == c
        ax.scatter(X_data[mask, 0], X_data[mask, 1],
                   s=14, alpha=0.45, color=palette[c])

        # Восстанавливаем ковариационную матрицу для отрисовки эллипса.
        if ct == "full":
            cov = m.covariances_[c]
        elif ct == "tied":
            cov = m.covariances_
        elif ct == "diag":
            cov = np.diag(m.covariances_[c])
        else:  # spherical
            cov = np.eye(2) * m.covariances_[c]

        draw_ellipse(ax, m.means_[c], cov, palette[c])

    ax.scatter(m.means_[:, 0], m.means_[:, 1],
               s=160, color="white", edgecolors=VIZ["node_text"],
               linewidths=2, zorder=5, marker="*")
    ax.set_title(f"{COV_LABELS[ct]}\nBIC = {bic_variants[ct]:.0f}")
    ax.set_xlabel("Признак 1")
    ax.set_ylabel("Признак 2")

fig.suptitle("GMM: формы компонент при разных типах ковариации (K=3)",
             fontsize=15, fontweight="semibold", y=1.01)
fig.tight_layout()
plt.show()

## 5. Интерпретация результата

- **Оценённые средние** (`gmm.means_`) — положение центра каждой компоненты в пространстве признаков. Если данные интерпретируемы, это «типичный представитель» подпопуляции.
- **Весовые доли** (`gmm.weights_`) — относительный размер каждой подпопуляции. Например, доля 0.35 означает, что ~35% наблюдений порождены этой компонентой.
- **Ковариационные матрицы** (`gmm.covariances_`) — форма, ориентация и разброс каждого облака. Диагональные элементы — дисперсии признаков внутри компоненты; внедиагональные — ковариации (корреляции).
- **Responsibility** (`predict_proba`) — вектор вероятностей для каждого наблюдения. Значения, далёкие от 0 и 1, указывают на «пограничные» точки: их интерпретация требует осторожности.
- **Логарифмическое правдоподобие** (`lower_bound_`) — качество подгонки в единицах вероятности. При сравнении нескольких моделей (разные K или типы ковариации) выбирайте по **BIC**, не по сырому правдоподобию.
- **BIC** — инструмент выбора K, но не гарантия. Найденное число компонент не обязано совпадать с числом содержательных групп в предметной области; один феномен может описываться несколькими компонентами.

**Предупреждения практику:**

- При малой выборке (меньше нескольких десятков наблюдений на компоненту) EM сходится ненадёжно; предпочтительны более ограниченные типы ковариации (`diag` или `spherical`).
- Если некоторые компоненты получили нулевые или почти нулевые веса — уменьшите K.
- В высокой размерности (сотни признаков) применяйте GMM только после снижения размерности (PCA до 10–30 компонент).

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными. Подготовьте таблицу числовых признаков (строки — наблюдения, столбцы — признаки); метки подпопуляций не нужны.

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже.
3. Запустите ячейки разделов 4 и 5 — код переиспользуется без изменений.

In [ ]:
# --- Шаблон загрузки своих данных ---
# import pandas as pd
# from sklearn.preprocessing import StandardScaler
#
# df = pd.read_csv("my_data.csv")           # ваш файл
# id_columns = []                            # столбцы-идентификаторы (не признаки)
#
# X_data = df.drop(columns=id_columns).select_dtypes("number").to_numpy()
#
# # Стандартизация рекомендована, если признаки имеют разный масштаб.
# X_data = StandardScaler().fit_transform(X_data)
#
# print(f"Загружено {X_data.shape[0]} наблюдений, {X_data.shape[1]} признаков.")
# # Далее повторите ячейки разделов 4 и 5:
# # подберите K по BIC, запустите GMM с best_k_bic компонентами,
# # получите responsibilities и интерпретируйте оценённые средние.

## Поэкспериментируйте сами

На демонстрационном наборе попробуйте:

| Что изменить | Что наблюдать |
|---|---|
| `n_components=5` при подгонке GMM | Появятся ли «лишние» компоненты с малыми весами? |
| `covariance_type="spherical"` вместо `"full"` | Как изменится BIC? Насколько хуже восстанавливаются наклонённые облака? |
| `n_init=1` вместо `n_init=10` | Увеличится ли разброс правдоподобий при повторных запусках? |
| `TRUE_WEIGHTS = [0.5, 0.45, 0.05]` (сильная несбалансированность) | Сможет ли BIC найти три компоненты при доле 5%? |
| Добавить четвёртое облако в `TRUE_MEANS` | Как кривая BIC сдвигает минимум? |

## Краткие выводы

- Смесевая модель (GMM) представляет данные как взвешенную сумму нескольких гауссовых компонент — каждая описывает одну скрытую подпопуляцию.
- EM-алгоритм чередует E-шаг (вычисление responsibility) и M-шаг (обновление параметров). Логарифмическое правдоподобие монотонно растёт, но сходимость гарантирована лишь к локальному оптимуму — используйте несколько рестартов (`n_init`).
- GMM даёт мягкую кластеризацию: каждое наблюдение получает вектор вероятностей принадлежности. Это честнее жёсткого разбиения K-средних при перекрывающихся компонентах.
- Число компонент K выбирают по минимуму BIC; BIC консервативнее AIC и реже переоценивает K на больших выборках.
- Тип ковариации определяет геометрию компонент: `full` — наиболее гибкий, но требует больше данных; `spherical` — аналог гауссова K-средних.
- Перед применением к многомерным данным снизьте размерность методом PCA: ковариационные матрицы `full` содержат O(d²) параметров на компоненту.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Вы применили GMM с `n_init=1` и получили определённый результат. Коллега запустил тот же код с другим `random_state` и получил другие веса компонент и другое значение правдоподобия. Что это означает и как следует поступить?

2. Кривая BIC при увеличении K не имеет выраженного минимума: она убывает, а затем выходит на плато при K = 4–6. Как выбрать число компонент в этой ситуации?

3. GMM показала ответственность 0.51 / 0.49 для одного из наблюдений между двумя компонентами. Исследователь присвоил этому наблюдению жёсткую метку первой компоненты (argmax). Корректно ли это и какую альтернативу стоит рассмотреть?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Это проявление локальных оптимумов EM-алгоритма: разные начальные точки привели к разным решениям. Корректный подход — использовать `n_init` не менее 5–10 и выбирать модель с наибольшим логарифмическим правдоподобием среди всех рестартов. scikit-learn делает это автоматически при `n_init > 1`.

2. Плато BIC свидетельствует о том, что данные не имеют единственного выраженного числа подпопуляций — или что компоненты при K > 3 содержательно неразличимы. В этой ситуации: (а) опираться на предметные знания (сколько биологически обоснованных групп ожидается?); (б) проверить устойчивость разбиения при K из диапазона плато; (в) выбрать наименьшее K на плато как наиболее парсимоничное.

3. Присвоение жёсткой метки через argmax при вероятностях 0.51/0.49 — искусственное: модель в равной мере допускает оба варианта. Альтернативы: (а) сохранять само значение responsibility как признак для дальнейшего анализа; (б) выделять такие наблюдения в отдельную группу «неопределённых» и анализировать их отдельно; (в) принимать решения о принадлежности только при уверенности выше заданного порога (например, 0.80).
</details>